## A1 – Create SQLite Database from Raw Data

First, we read the Excel file, skip the title row, load it into a SQLite table, and verify the schema and row count.

In [8]:
import pandas as pd
import sqlite3

# Read raw data – skip the first row which contains the title "DAY 90 — RAW DATA ..."
df = pd.read_excel('Day90_SQL_Python_Pipeline.xlsx', sheet_name='Raw Data', skiprows=1)

# Create / connect to SQLite database (file will be created if missing)
conn = sqlite3.connect('techmart.db')

# Write DataFrame to 'orders' table; replace if table already exists
df.to_sql('orders', conn, if_exists='replace', index=False)

# Verify table schema using PRAGMA
schema = pd.read_sql("PRAGMA table_info(orders)", conn)
print("Table schema:")
print(schema[['name', 'type']])

# Verify row count
row_count = pd.read_sql("SELECT COUNT(*) AS total_rows FROM orders", conn)
print(f"\nTotal rows loaded: {row_count['total_rows'][0]}")

Table schema:
         name     type
0    order_id     TEXT
1  order_date     TEXT
2         rep     TEXT
3        city     TEXT
4       state     TEXT
5    category     TEXT
6     product     TEXT
7      amount  INTEGER
8    quantity  INTEGER
9      status     TEXT

Total rows loaded: 150


## A2 – Four SQL Queries via pandas.read_sql()

All queries exclude cancelled orders to reflect true business performance.

In [9]:
# Q1: Total revenue by category (revenue = amount * quantity)
q1 = pd.read_sql('''
    SELECT 
        category,
        SUM(amount * quantity) AS revenue
    FROM orders
    WHERE status != 'Cancelled'
    GROUP BY category
    ORDER BY revenue DESC
''', conn)

print("=== Q1: Revenue by Category ===")
print(q1)

# Q2: Top 3 reps by total revenue
q2 = pd.read_sql('''
    SELECT 
        rep,
        SUM(amount * quantity) AS revenue,
        COUNT(*) AS order_count
    FROM orders
    WHERE status != 'Cancelled'
    GROUP BY rep
    ORDER BY revenue DESC
    LIMIT 3
''', conn)

print("\n=== Q2: Top 3 Reps ===")
print(q2)

# Q3: Monthly revenue trend (using strftime to extract month)
q3 = pd.read_sql('''
    SELECT 
        strftime('%m', order_date) AS month,
        SUM(amount * quantity) AS revenue,
        COUNT(*) AS order_count
    FROM orders
    WHERE status != 'Cancelled'
    GROUP BY month
    ORDER BY month
''', conn)

print("\n=== Q3: Monthly Revenue Trend ===")
print(q3)

# Q4: City-wise average order value (AOV) where AOV > 5000
q4 = pd.read_sql('''
    SELECT 
        city,
        ROUND(AVG(amount * quantity), 0) AS avg_order_value,
        COUNT(*) AS order_count
    FROM orders
    WHERE status != 'Cancelled'
    GROUP BY city
    HAVING avg_order_value > 5000
    ORDER BY avg_order_value DESC
''', conn)

print("\n=== Q4: Cities with AOV > 5000 ===")
print(q4)

=== Q1: Revenue by Category ===
         category  revenue
0     Electronics  3083813
1  Home & Kitchen   557429
2        Clothing   138863
3           Books    61615

=== Q2: Top 3 Reps ===
      rep  revenue  order_count
0  Kavita   867941           16
1   Sneha   722365           19
2   Arjun   570369           17

=== Q3: Monthly Revenue Trend ===
  month  revenue  order_count
0    01   866985           22
1    02   820420           26
2    03   598108           24
3    04   468408           17
4    05   570339           26
5    06   517460           21

=== Q4: Cities with AOV > 5000 ===
        city  avg_order_value  order_count
0     Mumbai          34501.0           16
1      Delhi          33332.0           17
2    Kolkata          30016.0           20
3       Pune          29724.0           16
4  Hyderabad          28654.0           21
5     Jaipur          27179.0           20
6  Bangalore          19612.0           10
7    Chennai          19107.0           16


## A3 – Parameterised Query Function

Uses `?` placeholder to prevent SQL injection. The function is fully reusable for any city.

In [11]:
def get_city_report(conn, city_name):
    """
    Returns a DataFrame aggregated by rep and category for a given city.
    Only non-cancelled orders are considered.
    """
    query = '''
        SELECT 
            rep,
            category,
            SUM(amount * quantity) AS total_revenue,
            COUNT(*) AS order_count
        FROM orders
        WHERE city = ? AND status != 'Cancelled'
        GROUP BY rep, category
        ORDER BY total_revenue DESC
    '''
    return pd.read_sql(query, conn, params=[city_name])

# Test with Mumbai and Delhi
print("=== Mumbai Report ===")
print(get_city_report(conn, 'Mumbai'))

print("\n=== Delhi Report ===")
print(get_city_report(conn, 'Delhi'))

=== Mumbai Report ===
       rep        category  total_revenue  order_count
0    Rohit     Electronics         136268            1
1    Priya     Electronics          89544            1
2    Sneha     Electronics          88702            2
3   Kavita     Electronics          82264            1
4    Rahul     Electronics          76431            2
5    Meera     Electronics          31788            1
6    Meera  Home & Kitchen          19275            1
7   Vikram  Home & Kitchen           9946            1
8    Rohit        Clothing           6291            1
9    Rahul        Clothing           5512            1
10   Priya           Books           3185            1
11   Rohit           Books           1280            1
12  Vikram           Books            774            1
13   Meera        Clothing            762            1

=== Delhi Report ===
       rep        category  total_revenue  order_count
0    Sneha     Electronics         168576            1
1   Kavita     Electr

## A4 – Transform Q1 Results with Pandas

Add % share, rank, status label, and format revenue for readability.

In [12]:
# Start from Q1 result
df_cat = q1.copy()

# Calculate percentage share of total revenue
total_rev = df_cat['revenue'].sum()
df_cat['% Share'] = (df_cat['revenue'] / total_rev * 100).round(1)

# Rank by revenue (1 = highest)
df_cat['Rank'] = df_cat['revenue'].rank(ascending=False).astype(int)

# Assign status based on rank and share
def assign_status(row):
    if row['Rank'] == 1:
        return 'Leader'
    elif row['% Share'] > 20:
        return 'Growth'
    else:
        return 'Watch'

df_cat['Status'] = df_cat.apply(assign_status, axis=1)

# Create a formatted revenue string for export
df_cat['revenue_fmt'] = df_cat['revenue'].apply(lambda x: f"₹{x:,.0f}")

# Display transformed DataFrame
print(df_cat[['category', 'revenue', '% Share', 'Rank', 'Status']])

         category  revenue  % Share  Rank  Status
0     Electronics  3083813     80.3     1  Leader
1  Home & Kitchen   557429     14.5     2   Watch
2        Clothing   138863      3.6     3   Watch
3           Books    61615      1.6     4   Watch


## A5 – Export Results to Formatted Excel with openpyxl

Creates a workbook with three sheets: Revenue by Category, Top Reps, and Monthly Trend.  
Applies dark blue headers, alternating row colours, and currency formatting.

In [13]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment

wb = Workbook()

# ---------- helper functions ----------
def style_header(ws, row, headers):
    """Writes header row with dark blue background and white bold text."""
    for col_idx, header in enumerate(headers, 1):
        cell = ws.cell(row=row, column=col_idx, value=header)
        cell.font = Font(name='Arial', bold=True, color='FFFFFF', size=10)
        cell.fill = PatternFill('solid', fgColor='1F3864')
        cell.alignment = Alignment(horizontal='center')

def write_data(ws, start_row, df, fmt_cols=None):
    """Writes DataFrame rows starting at start_row with alternating row colours."""
    for i, (_, row_data) in enumerate(df.iterrows()):
        row_num = start_row + i
        bg_color = 'FFFFFF' if i % 2 == 0 else 'F8F9FA'
        for j, value in enumerate(row_data, 1):
            cell = ws.cell(row=row_num, column=j, value=value)
            cell.font = Font(name='Arial', size=10)
            cell.fill = PatternFill('solid', fgColor=bg_color)
            # Apply number format to numeric columns if specified
            if fmt_cols and j in fmt_cols:
                cell.number_format = '#,##0'

# ---------- Sheet 1: Revenue by Category ----------
ws_cat = wb.active
ws_cat.title = 'Revenue by Category'
style_header(ws_cat, 1, ['Category', 'Revenue', '% Share', 'Rank', 'Status'])
# Prepare dataframe with only the required columns
cat_export = df_cat[['category', 'revenue', '% Share', 'Rank', 'Status']]
write_data(ws_cat, 2, cat_export, fmt_cols={2})  # revenue column formatted

# ---------- Sheet 2: Top Reps ----------
ws_rep = wb.create_sheet('Top Reps')
style_header(ws_rep, 1, ['Rep', 'Revenue', 'Orders'])
rep_export = q2[['rep', 'revenue', 'order_count']]
write_data(ws_rep, 2, rep_export, fmt_cols={2})

# ---------- Sheet 3: Monthly Trend ----------
ws_mon = wb.create_sheet('Monthly Trend')
style_header(ws_mon, 1, ['Month', 'Revenue', 'Orders'])
mon_export = q3[['month', 'revenue', 'order_count']]
write_data(ws_mon, 2, mon_export, fmt_cols={2})

# Save workbook
wb.save('Day90_Pipeline_Output.xlsx')
print("✅ Formatted Excel output saved as 'Day90_Pipeline_Output.xlsx'")

✅ Formatted Excel output saved as 'Day90_Pipeline_Output.xlsx'


## A6 – Full Pipeline as One Reusable Function

Combines all steps: read input → load SQLite → run queries → transform → export.  
Includes error handling and progress messages.

In [14]:
def run_weekly_report(input_file, output_file):
    """
    Automates the weekly sales reporting pipeline.
    input_file : CSV or Excel file with raw order data
    output_file: Path to the formatted Excel report
    """
    import pandas as pd
    import sqlite3
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment

    try:
        # ---------- Step 1: Read input ----------
        print('[1/5] Reading input data...')
        if input_file.endswith('.csv'):
            df = pd.read_csv(input_file)
        else:
            # Excel file with title row – skip it
            df = pd.read_excel(input_file, sheet_name='Raw Data', skiprows=1)
        print(f'       {len(df)} rows loaded.')

        # ---------- Step 2: Load to SQLite ----------
        print('[2/5] Loading data into SQLite...')
        conn = sqlite3.connect(':memory:')   # use in-memory DB for speed
        df.to_sql('orders', conn, if_exists='replace', index=False)

        # ---------- Step 3: Run queries ----------
        print('[3/5] Running SQL queries...')
        q_cat = pd.read_sql('''
            SELECT category, SUM(amount * quantity) AS revenue
            FROM orders WHERE status != "Cancelled"
            GROUP BY category ORDER BY revenue DESC
        ''', conn)

        q_rep = pd.read_sql('''
            SELECT rep, SUM(amount * quantity) AS revenue, COUNT(*) AS orders
            FROM orders WHERE status != "Cancelled"
            GROUP BY rep ORDER BY revenue DESC LIMIT 3
        ''', conn)

        q_mon = pd.read_sql('''
            SELECT strftime('%m', order_date) AS month, SUM(amount * quantity) AS revenue,
                   COUNT(*) AS orders
            FROM orders WHERE status != "Cancelled"
            GROUP BY month ORDER BY month
        ''', conn)

        # ---------- Step 4: Transform ----------
        print('[4/5] Transforming results...')
        q_cat['% Share'] = ((q_cat['revenue'] / q_cat['revenue'].sum()) * 100).round(1)
        q_cat['Rank'] = q_cat['revenue'].rank(ascending=False).astype(int)
        # Status column (same logic as A4)
        def status(row):
            if row['Rank'] == 1: return 'Leader'
            elif row['% Share'] > 20: return 'Growth'
            else: return 'Watch'
        q_cat['Status'] = q_cat.apply(status, axis=1)

        # ---------- Step 5: Export to formatted Excel ----------
        print('[5/5] Exporting to Excel...')
        wb = Workbook()

        def style_header(ws, row, headers):
            for i, h in enumerate(headers, 1):
                c = ws.cell(row=row, column=i, value=h)
                c.font = Font(name='Arial', bold=True, color='FFFFFF', size=10)
                c.fill = PatternFill('solid', fgColor='1F3864')
                c.alignment = Alignment(horizontal='center')

        def write_data(ws, start_row, df, money_col=None):
            for i, (_, row_data) in enumerate(df.iterrows()):
                bg = 'FFFFFF' if i % 2 == 0 else 'F8F9FA'
                for j, val in enumerate(row_data, 1):
                    c = ws.cell(row=start_row + i, column=j, value=val)
                    c.font = Font(name='Arial', size=10)
                    c.fill = PatternFill('solid', fgColor=bg)
                    if money_col and j == money_col:
                        c.number_format = '#,##0'

        # Sheet 1
        ws1 = wb.active
        ws1.title = 'Revenue by Category'
        style_header(ws1, 1, ['Category', 'Revenue', '% Share', 'Rank', 'Status'])
        write_data(ws1, 2, q_cat[['category', 'revenue', '% Share', 'Rank', 'Status']], money_col=2)

        # Sheet 2
        ws2 = wb.create_sheet('Top Reps')
        style_header(ws2, 1, ['Rep', 'Revenue', 'Orders'])
        write_data(ws2, 2, q_rep, money_col=2)

        # Sheet 3
        ws3 = wb.create_sheet('Monthly Trend')
        style_header(ws3, 1, ['Month', 'Revenue', 'Orders'])
        write_data(ws3, 2, q_mon, money_col=2)

        wb.save(output_file)
        conn.close()
        print(f'✅ Pipeline complete! Output saved to {output_file}')

    except Exception as e:
        print(f'❌ Pipeline failed: {e}')

# Test the full pipeline
run_weekly_report('Day90_SQL_Python_Pipeline.xlsx', 'Day90_Pipeline_Output.xlsx')

[1/5] Reading input data...
       150 rows loaded.
[2/5] Loading data into SQLite...
[3/5] Running SQL queries...
[4/5] Transforming results...
[5/5] Exporting to Excel...
✅ Pipeline complete! Output saved to Day90_Pipeline_Output.xlsx


## BONUS – NRA (Number – Reason – Action) Insight

From the pipeline output, we extract one actionable insight based on real numbers.

In [15]:
# Find the top category from Q1
top_cat = df_cat.loc[df_cat['Rank'] == 1, 'category'].values[0]
top_rev = df_cat.loc[df_cat['Rank'] == 1, 'revenue'].values[0]

print(f"₹{top_rev:,.0f} — {top_cat} is the highest revenue category — "
      f"Action: increase cross‑sell by bundling {top_cat} with growing categories "
      f"(like Books) to raise overall basket size.")

₹3,083,813 — Electronics is the highest revenue category — Action: increase cross‑sell by bundling Electronics with growing categories (like Books) to raise overall basket size.
